# Transcripción de audios del panel con Whisper

Panel WhatsApp — Encuesta Permanente CISCo-IJD

Este notebook es el paso del medio de `code/transcripciones.qmd`:

1. R descarga los `.ogg` de la ronda y los sube a **`A/<ronda>`** en Drive.
2. **Este notebook** lee esa carpeta, transcribe y deja un `.txt` por audio en **`B/<ronda>`**.
3. R baja los `.txt` y reemplaza los enlaces en el CSV.

> **Regla que no se puede romper:** el `.txt` tiene que llamarse igual que el `.ogg`
> (`q3_fila57.ogg` → `q3_fila57.txt`). R usa ese nombre para saber a qué fila y a qué
> pregunta corresponde la transcripción.

**Antes de correr:** Entorno de ejecución → Cambiar tipo de entorno → GPU (T4).
Sin GPU funciona, pero tarda del orden de 10 veces más.

In [ ]:
#@title 1) Instalar dependencias
!pip install -q faster-whisper

import torch
print("GPU disponible:", torch.cuda.is_available(),
      "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

In [ ]:
#@title 2) Montar Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
#@title 3) Configuración de la ronda

# Tienen que coincidir con project.yml del proyecto en R:
#   ronda$round_id, drive$folder_audio, drive$folder_text
ROUND_ID     = "R1"      #@param {type:"string"}
FOLDER_AUDIO = "A"       #@param {type:"string"}
FOLDER_TEXT  = "B"       #@param {type:"string"}
MODELO       = "large-v3" #@param ["large-v3", "medium", "small"]
IDIOMA       = "es"      #@param {type:"string"}

from pathlib import Path

DRIVE = Path('/content/drive/MyDrive')
dir_audio = DRIVE / FOLDER_AUDIO / ROUND_ID
dir_text  = DRIVE / FOLDER_TEXT  / ROUND_ID
dir_text.mkdir(parents=True, exist_ok=True)

assert dir_audio.exists(), f"No existe {dir_audio}. Crear la carpeta y subir los audios desde R."

audios = sorted(dir_audio.glob('*.ogg'))
hechos = {p.stem for p in dir_text.glob('*.txt')}
pendientes = [p for p in audios if p.stem not in hechos]

print(f"audios en {dir_audio}: {len(audios)}")
print(f"ya transcritos: {len(hechos)} | pendientes: {len(pendientes)}")

In [ ]:
#@title 4) Cargar el modelo
from faster_whisper import WhisperModel

compute_type = "float16" if torch.cuda.is_available() else "int8"
device       = "cuda" if torch.cuda.is_available() else "cpu"

model = WhisperModel(MODELO, device=device, compute_type=compute_type)
print(f"modelo {MODELO} cargado en {device} ({compute_type})")

In [ ]:
#@title 5) Transcribir
import time

errores = []
t0 = time.time()

for i, p in enumerate(pendientes, 1):
    destino = dir_text / f"{p.stem}.txt"
    try:
        segmentos, info = model.transcribe(
            str(p),
            language=IDIOMA,
            vad_filter=True,          # recorta silencios, evita alucinaciones en audios cortos
            beam_size=5,
        )
        texto = " ".join(s.text.strip() for s in segmentos).strip()
        # el .txt se escribe siempre, incluso vacío, para no reprocesar audios sin voz
        destino.write_text(texto, encoding='utf-8')
        print(f"[{i}/{len(pendientes)}] {p.name} ({info.duration:.0f}s) -> {texto[:70]!r}")
    except Exception as e:
        errores.append((p.name, str(e)))
        print(f"[{i}/{len(pendientes)}] ERROR en {p.name}: {e}")

print(f"\nlisto en {(time.time()-t0)/60:.1f} min | errores: {len(errores)}")
for n, e in errores:
    print(" -", n, "|", e)

In [ ]:
#@title 6) Control final
txts = sorted(dir_text.glob('*.txt'))
vacios = [p.name for p in txts if not p.read_text(encoding='utf-8').strip()]
faltan = [p.name for p in audios if not (dir_text / f"{p.stem}.txt").exists()]

print(f"audios: {len(audios)} | txt generados: {len(txts)}")
print(f"txt vacíos (audio sin voz detectada): {len(vacios)}")
if vacios: print("  ", vacios[:20])
print(f"audios sin transcribir: {len(faltan)}")
if faltan: print("  ", faltan[:20])

print("\nSi no falta ninguno, volver a R y correr el bloque 5 de code/transcripciones.qmd")